# Open coding, then focused coding

This notebook is the short form of [methodology.md](methodology.md). It does not call a provider. The default review reads **every document** in a seeded order and **every passage** of that document in reading order (Glaser and Strauss 1967, constant comparison; Charmaz 2006/2014, incident coding).

Sources used by the pipeline:

- Glaser, B. G., & Strauss, A. L. (1967). *The Discovery of Grounded Theory*. Aldine. https://doi.org/10.4324/9780203793206
- Glaser, B. G. (1978). *Theoretical Sensitivity*. Sociology Press.
- Glaser, B. G. (1992). *Basics of Grounded Theory Analysis: Emergence vs. Forcing*. Sociology Press. The context sensitizes. It is not a codebook.
- Strauss, A., & Corbin, J. (1990/1998). *Basics of Qualitative Research*. Sage. Properties and dimensions belong in memos. Axial coding is out of scope.
- Charmaz, K. (2006/2014). *Constructing Grounded Theory*. Sage. Gerunds, in-vivo codes, memos, then focused coding. Saturation is not declared by the software.
- Chen, J., et al. https://arxiv.org/abs/2411.12142
- JMIR (2025). https://www.jmir.org/2025/1/e70122


## Review modes

`review_mode="document"` (default): codebook updates after the document, or after each passage when `batch_size=1`. `pipeline=True` overlaps synthesis of the finished document with coding of the next one, then applies the merge map in chunk order.

`review_mode="wave"` is opt-in. **It weakens constant comparison.** Passages from several documents can share one frozen codebook, and a later passage in a document can be coded before an earlier one. `parallel_documents=True` weakens comparison further.

A fake-model timing run (0.05s sleep per call, 24 passages) is in the README and in methodology.md: sequential 1.588s / 28 calls, document mode 0.525s / 10 calls, wave mode 0.315s / 9 calls. All three ended with 3 codes and no leftover near-duplicate pair on that fake coder. Wave was fastest because it mixed documents.


In [ ]:
from automated_grounded_theory import open_code, build_themes, plan_coding

corpus = [
    {"id": "alex", "text": "I spent the morning packing my suitcase."},
    {"id": "blair", "text": "Packing bags up with boxes took all night."},
]
context = {
    "research_question": "How do students describe the days around leaving home?",
    "sensitizing_concepts": ["leaving home"],
}

plan = plan_coding(corpus, seed=7)
print("document order:", [item["doc_id"] for item in sorted(plan.passages, key=lambda item: item["coding_order"])])
print("Pass a chat model as model= to open_code. This cell does not.")


## Import into Taguette

Taguette (https://gitlab.com/remram44/taguette) imports a codebook CSV. It does **not** import highlights or `.qdc`.

1. Run `open_code(..., output_dir="coding_run")` and, after that, `build_themes(..., output_dir="coding_run/themes")`.
2. In Taguette, open the project and choose **Project info → Import codebook**.
3. For open codes, choose `coding_run/taguette_codebook.csv`. The header is `tag,description,number of highlights,number of documents`, which is Taguette's own export header. The importer (`taguette.import_codebook.list_tags`) uses `tag` and `description` and ignores the counts.
4. For themes, import `coding_run/themes/taguette_themes.csv` into a project (or the same project if you want the theme tree instead of the flat codes). Each code path is `Theme\Code`. A parent row is the theme name. Taguette treats the backslash as the parent/child split (`interesting\places` in its tests). Dots in a label are kept. A backslash inside a label is replaced so it does not add a level.
5. `codebook.qdc` and `taguette_themes.qdc` follow Taguette's REFI-QDA export (`CodeBook`, `Code` with `guid`, `name`, `isCodable="true"`) plus a `Description` child for other QDA packages. Taguette writes `.qdc` and does not import it.
6. `highlights.csv` matches Taguette's highlight export columns. Taguette does not import highlights; apply tags in Taguette if you need them stored there.

The sync functions `open_code` and `build_themes` can be called from this notebook. They run on a worker thread when a cell already has an event loop. Prefer `await aopen_code(...)` if you built the chat model on the notebook's loop.
